# Which parameters matter? An introduction to Sobol sensitivity analysis

A battery model has dozens of parameters, and many of them are uncertain: they are measured with some error, fitted to data, or taken from a paper about a slightly different cell. Two questions come up again and again:

- *Which of these uncertain parameters actually change the result I care about?*
- *Which ones can I safely fix at a nominal value, for example before a calibration?*

**Global sensitivity analysis** answers these questions, and **Sobol indices** are its most widely used tool. This notebook explains them from the ground up and shows how BattMo computes them. No prior knowledge of sensitivity analysis is assumed; some familiarity with running a BattMo `Simulation` helps (see the beginner tutorials).

**Contents**

1. Why *global* sensitivity analysis?
2. The idea in one picture, on a cheap test function
3. First-order and total-order Sobol indices
4. Method 1: Monte Carlo estimation with the Saltelli scheme
5. Method 2: Polynomial chaos expansion (PCE)
6. A sensitivity analysis of a battery discharge
7. Practical tips and pitfalls
8. Summary, references and further reading

Sections 2 to 5 use a small mathematical test function, so they run in seconds and we can compare every estimate with the exact answer. Section 6 runs a few hundred battery simulations and takes several minutes; start Julia with several threads (`julia -t auto`, or a Jupyter kernel with threads enabled) to speed it up.

> **Requirements:** a BattMo version that includes the sensitivity module (`SobolSensitivity`), and GLMakie for the plots.

## 1. Why *global* sensitivity analysis?

The most intuitive way to test a parameter is to change it a little, rerun the model, and see how much the result moves. This is a **local** sensitivity: a derivative at one point in parameter space. A variant is the **one-at-a-time** study, where each parameter is moved through its range while all others stay at their nominal values.

Both are useful, but they can mislead when the model is nonlinear:

- **The answer depends on where you look.** A parameter that hardly matters at the nominal point can matter a lot elsewhere in its plausible range, for example a diffusion coefficient that only becomes limiting when it is small.
- **They miss interactions.** Sometimes a parameter only has an effect when *another* parameter is also at an extreme value. Moving one parameter at a time never sees this.

A **global** sensitivity analysis moves *all* uncertain parameters *at the same time* over their *whole* plausible ranges, and asks how the resulting variation of the output can be attributed to each parameter. Sobol indices do this attribution using the *variance* of the output [Sobol' 2001].

## 2. The idea in one picture

### Parameters as random variables

The first step is to describe our uncertainty: for each parameter we choose a range (lower and upper bound) and assume that every value in the range is equally plausible, a *uniform distribution*. The parameters are assumed independent of each other.

If the inputs are random, the output $Y$ is random too. Its **variance** $\operatorname{Var}(Y)$, the average squared distance from its mean, measures how much $Y$ varies over all plausible parameter combinations. Sensitivity analysis asks:

> *Which share of $\operatorname{Var}(Y)$ is caused by each parameter?*

### A test function with a known answer

Before spending hours of simulation time on a battery, we use the **Ishigami function** [Ishigami & Homma 1990], a standard benchmark for sensitivity methods:

$$
Y = \sin X_1 + a \sin^2 X_2 + b\, X_3^4 \sin X_1, \qquad X_1, X_2, X_3 \sim \mathcal{U}(-\pi, \pi), \quad a = 7,\ b = 0.1.
$$

It has three useful properties: it is nonlinear, it contains an *interaction* ($X_3$ only appears multiplied by a function of $X_1$), and its exact Sobol indices are known analytically.

BattMo works internally with parameters scaled to the *unit cube* $[0, 1]^d$, and maps them to the physical bounds when a simulation is set up. We do the same here: the function takes $u \in [0, 1]^3$ and maps it to $[-\pi, \pi]^3$.

In [ ]:
using BattMo
using GLMakie
using Random
using Statistics
using Printf

In [ ]:
const a, b = 7.0, 0.1

# The Ishigami function on the unit cube: u ∈ [0, 1]^3 is mapped to x ∈ [-π, π]^3.
function ishigami(u)
    x = π .* (2 .* u .- 1)
    return sin(x[1]) + a * sin(x[2])^2 + b * x[3]^4 * sin(x[1])
end

# The exact Sobol indices, from the variance decomposition of the function.
V1 = 0.5 * (1 + b * π^4 / 5)^2       # variance due to X1 alone
V2 = a^2 / 8                         # variance due to X2 alone
V13 = b^2 * π^8 * (1 / 18 - 1 / 50)  # variance due to the interaction of X1 and X3
V = V1 + V2 + V13                    # total variance

exact_S1 = [V1, V2, 0.0] ./ V
exact_ST = [V1 + V13, V2, V13] ./ V

println("Parameter   exact S1   exact ST")
for i in 1:3
    @printf("X%d          %8.4f   %8.4f\n", i, exact_S1[i], exact_ST[i])
end

### Looking at scatter plots

Let us simply sample many random parameter combinations, evaluate the function, and plot $Y$ against each input. The red line is the average of $Y$ in thin vertical slices of the input: an estimate of the **conditional mean** $\mathbb{E}[Y \mid X_i]$, the average output *if we knew* $X_i$ exactly.

In [ ]:
rng = Random.Xoshiro(1)
n_scatter = 20_000
U = rand(rng, n_scatter, 3)                 # one row per random parameter combination
Y = [ishigami(row) for row in eachrow(U)]

# Average of y in `nbins` equally wide slices of x ∈ [0, 1]: an estimate of E[Y | X_i].
function binned_means(x, y; nbins = 40)
    bin = clamp.(ceil.(Int, x .* nbins), 1, nbins)
    centers = ((1:nbins) .- 0.5) ./ nbins
    return centers, [mean(y[bin .== k]) for k in 1:nbins]
end

fig_scatter_toy = Figure(size = (1000, 320))
for i in 1:3
    ax = Axis(fig_scatter_toy[1, i]; xlabel = "u$i (X$i scaled to [0, 1])", ylabel = i == 1 ? "Y" : "")
    scatter!(ax, U[:, i], Y; color = (:steelblue, 0.15), markersize = 3)
    centers, means = binned_means(U[:, i], Y)
    lines!(ax, centers, means; color = :red, linewidth = 3)
end
fig_scatter_toy

What do we see?

- **$X_1$ and $X_2$:** the red curve moves up and down considerably. Knowing $X_1$ (or $X_2$) shifts the expected output, so each of them explains a part of the variance on its own.
- **$X_3$:** the red curve is flat. Knowing $X_3$ alone does not change the *average* output at all. But the *spread* of the cloud is much larger near the ends of the range than in the middle! $X_3$ does matter, but only in combination with $X_1$: through the term $b X_3^4 \sin X_1$, a large $|X_3|$ amplifies whatever $\sin X_1$ is doing. This is an **interaction**.

A one-at-a-time study with $X_1$ fixed at its nominal value $0$ (where $\sin X_1 = 0$) would conclude that $X_3$ has *no* effect at all. The global view shows otherwise.

## 3. First-order and total-order Sobol indices

The scatter plots suggest two ways of measuring the importance of a parameter $X_i$. Both are expressed as a fraction of the total variance, so they lie between 0 and 1.

### First-order index $S_i$: the effect of $X_i$ alone

$$
S_i = \frac{\operatorname{Var}\big(\mathbb{E}[Y \mid X_i]\big)}{\operatorname{Var}(Y)}
$$

In words: *how much does the red curve vary, compared to the whole cloud?* Equivalently, $S_i$ is the fraction of the variance that would disappear, on average, if we learned the true value of $X_i$ and nothing else. For $X_3$ above, the red curve is flat, so $S_3 = 0$.

### Total-order index $S_{T,i}$: the effect of $X_i$ including all interactions

$$
S_{T,i} = \frac{\mathbb{E}\big[\operatorname{Var}(Y \mid X_{\sim i})\big]}{\operatorname{Var}(Y)}
$$

Here $X_{\sim i}$ means *all parameters except* $X_i$. In words: *fix all other parameters at some values, vary only $X_i$, and measure how much $Y$ still varies; average this over all ways of fixing the others.* This is the fraction of the variance that would *remain* if we knew everything except $X_i$. It includes both the effect of $X_i$ alone and every interaction that involves $X_i$. For $X_3$ above, $S_{T,3} \approx 0.24$, although $S_3 = 0$.

### Where these indices come from

Sobol' showed that, for independent inputs, the variance of any (square-integrable) function can be split uniquely into contributions of single parameters, pairs, triples, and so on [Sobol' 2001]:

$$
\operatorname{Var}(Y) = \sum_i V_i + \sum_{i<j} V_{ij} + \sum_{i<j<k} V_{ijk} + \dots
$$

The first-order index is the single-parameter term, $S_i = V_i / \operatorname{Var}(Y)$, and the total-order index sums *every* term that contains $i$. For the Ishigami function the only nonzero terms are $V_1$, $V_2$ and $V_{13}$, which is exactly what the exact formulas in the code cell above use.

### How to read them

| Observation | Meaning |
| --- | --- |
| $S_{T,i} \approx 0$ | $X_i$ has no influence within its range: it can be fixed at its nominal value. |
| $S_i$ large | $X_i$ matters on its own. Pinning down its value reduces the uncertainty of $Y$ the most. |
| $S_{T,i} - S_i$ large | The effect of $X_i$ depends on the values of other parameters (interactions). |
| $\sum_i S_i \approx 1$ | The output is nearly *additive*: no interactions. |
| $\sum_i S_i \ll 1$ | Interactions are important. |

Always $0 \le S_i \le S_{T,i} \le 1$. Note also that **the indices depend on the chosen ranges**: widening the range of one parameter increases its share of the variance at the expense of the others. The ranges should reflect how uncertain each parameter really is.

### A first estimate by brute force

The first-order index is the variance of the red curve divided by the variance of the whole cloud, so we can estimate it directly from our 20 000 samples:

In [ ]:
println("Parameter   S1 (from the red curves)   exact S1")
for i in 1:3
    _, means = binned_means(U[:, i], Y)
    @printf("X%d          %8.3f                   %8.3f\n", i, var(means) / var(Y), exact_S1[i])
end

This works, but it needed 20 000 evaluations, it only gives the first-order index, and the slicing trick becomes useless with many parameters. A brute-force estimate of $S_{T,i}$ needs nested loops (for many fixed values of $X_{\sim i}$, many values of $X_i$), so millions of evaluations. For a function that takes microseconds that is fine; for a battery simulation that takes seconds it is not.

BattMo offers two smarter estimators, explained in the next two sections.

## 4. Method 1: Monte Carlo estimation with the Saltelli scheme

The Saltelli scheme [Saltelli et al. 2010] estimates *all* first- and total-order indices from a cleverly arranged set of random samples.

### The sampling design

1. Draw two independent random matrices $A$ and $B$, each with $N$ rows (sample points) and $d$ columns (parameters).
2. For each parameter $i$, build the matrix $A_B^{(i)}$: a copy of $A$ in which column $i$ is taken from $B$.
3. Evaluate the model at every row of $A$, $B$ and all $d$ matrices $A_B^{(i)}$.

For $d = 3$, the columns of the matrices come from:

| Matrix | column 1 | column 2 | column 3 |
| --- | --- | --- | --- |
| $A$ | A | A | A |
| $B$ | B | B | B |
| $A_B^{(1)}$ | **B** | A | A |
| $A_B^{(2)}$ | A | **B** | A |
| $A_B^{(3)}$ | A | A | **B** |

This needs $N(d + 2)$ model evaluations. BattMo builds these points with an internal function, which we call here to look at a tiny example with $N = 2$:

In [ ]:
A = round.(rand(Random.Xoshiro(2), 2, 3); digits = 2)
B = round.(rand(Random.Xoshiro(3), 2, 3); digits = 2)
# Rows: A (2 rows), B (2 rows), then A_B^(1), A_B^(2), A_B^(3) (2 rows each).
BattMo.saltelli_points(A, B)

### Why this works

The trick is in which rows share which values.

- **Total order.** Row $j$ of $A$ and row $j$ of $A_B^{(i)}$ are identical *except* for $X_i$. Their difference therefore shows how much $Y$ changes when only $X_i$ changes, with all other parameters held fixed: exactly what $S_{T,i}$ measures. The estimator of Jansen [Jansen 1999] averages the squared differences:

$$
S_{T,i} \approx \frac{1}{V} \frac{1}{2N} \sum_{j=1}^{N} \Big(f_A^{(j)} - f_{A_B^{(i)}}^{(j)}\Big)^2 .
$$

- **First order.** Row $j$ of $B$ and row $j$ of $A_B^{(i)}$ share *only* $X_i$. If the outputs at these two points are correlated, this can only be through $X_i$, and the strength of that correlation is $\operatorname{Var}(\mathbb{E}[Y \mid X_i])$:

$$
S_i \approx \frac{1}{V} \frac{1}{N} \sum_{j=1}^{N} \tilde f_B^{(j)} \Big(f_{A_B^{(i)}}^{(j)} - f_A^{(j)}\Big).
$$

  Subtracting $f_A$, which is independent of $B$, removes a large bias term. Here $f_A^{(j)}$ is the model output at row $j$ of $A$ (and similarly for the others), $V$ is the variance of all outputs at $A$ and $B$, and $\tilde f_B = f_B - \bar f$ is centred by the mean output. The centring does not change the average of the estimate, but it makes it much less noisy for quantities with a large mean, such as the energy of a discharge.

### Trying it on the Ishigami function

We now evaluate the function at the Saltelli points, and estimate the indices with the same internal function that BattMo's `solve` uses. The functions `saltelli_points`, `estimate_sobol_indices` and those used in section 5 are BattMo internals, not public API: we call them here only to show what happens inside. For your own studies, use `SobolSensitivity` as in section 6.

In [ ]:
d = 3
N = 1024
rng = Random.Xoshiro(42)
A, B = rand(rng, N, d), rand(rng, N, d)
points = BattMo.saltelli_points(A, B)              # N * (d + 2) = 5120 points
Y_saltelli = [ishigami(row) for row in eachrow(points)]

est = BattMo.estimate_sobol_indices(reshape(Y_saltelli, :, 1), N, d; bootstrap = 200, rng)

println("Saltelli, N = $N ($(N * (d + 2)) evaluations)")
println("Param   S1 estimate [95% CI]        exact    ST estimate [95% CI]        exact")
for i in 1:d
    @printf(
        "X%d     %6.3f [%6.3f, %6.3f]   %6.3f    %6.3f [%6.3f, %6.3f]   %6.3f\n",
        i, est.first_order[i], est.first_order_ci[1][i], est.first_order_ci[2][i], exact_S1[i],
        est.total_order[i], est.total_order_ci[1][i], est.total_order_ci[2][i], exact_ST[i],
    )
end

The estimates are close to the exact values, but even with 5120 evaluations they are not exact: $S_3$ comes out slightly above zero, and some exact values can fall just outside the confidence intervals. The bootstrap intervals are themselves only estimates of the uncertainty.

### Confidence intervals by bootstrapping

Monte Carlo estimates are random: another set of random samples gives slightly different numbers. To know how much to trust them without rerunning the model, BattMo uses the **bootstrap**: it resamples the $N$ rows (with replacement, so some rows appear twice and others not at all), recomputes the indices, repeats this many times, and reports the range that contains the middle 95% of these recomputed values. **Wide intervals mean that more samples are needed.** With few samples, an estimate can even fall slightly outside $[0, 1]$, or $S_i$ can exceed $S_{T,i}$ for unimportant parameters; this is sampling noise, not a bug.

### How fast does it converge?

Let us repeat the estimate for increasing $N$ and look at the average error compared to the exact indices.

In [ ]:
Ns = 2 .^ (5:13)
repetitions = 20
mean_error = map(Ns) do N
    errors = map(1:repetitions) do r
        rng = Random.Xoshiro(r)
        points = BattMo.saltelli_points(rand(rng, N, d), rand(rng, N, d))
        Y = [ishigami(row) for row in eachrow(points)]
        e = BattMo.estimate_sobol_indices(reshape(Y, :, 1), N, d)
        mean(abs, vcat(e.first_order .- exact_S1, e.total_order .- exact_ST))
    end
    mean(errors)
end

fig_convergence = Figure(size = (600, 400))
ax = Axis(
    fig_convergence[1, 1]; xscale = log10, yscale = log10,
    xlabel = "Number of model evaluations N (d + 2)", ylabel = "Mean absolute error of the indices",
    title = "Saltelli estimates of the Ishigami indices",
)
evaluations = Ns .* (d + 2)
scatterlines!(ax, evaluations, mean_error; label = "Saltelli (average of $repetitions runs)")
lines!(ax, evaluations, mean_error[1] .* sqrt.(evaluations[1] ./ evaluations); linestyle = :dash, color = :gray, label = "∝ 1/√N")
axislegend(ax)
fig_convergence

The error decreases like $1/\sqrt{N}$, the typical rate of Monte Carlo methods: **to halve the error, we need four times as many evaluations.** An error of about 0.01 needs tens of thousands of evaluations here. The method makes no assumption about the model, which makes it robust, but this is expensive when one evaluation is a battery simulation of several seconds.

## 5. Method 2: Polynomial chaos expansion (PCE)

The second method takes a different route [Sudret 2008]:

1. Run the model at a moderate number of points.
2. Fit a cheap **surrogate model**, a polynomial in the parameters, to these results.
3. Compute the Sobol indices of the polynomial *exactly*, directly from its coefficients.

If the polynomial is a good approximation of the model, its Sobol indices are good approximations of the model's indices. Despite the name, there is nothing chaotic about it: "polynomial chaos" is a historical term for a polynomial in random variables.

### Orthonormal polynomials

The key ingredient is the choice of polynomials. BattMo uses **Legendre polynomials** $\psi_0, \psi_1, \psi_2, \ldots$ (of degree 0, 1, 2, ...), shifted to $[0, 1]$ and scaled so that they are *orthonormal* for the uniform distribution:

$$
\mathbb{E}[\psi_m(U)\, \psi_n(U)] = \begin{cases} 1 & m = n \\ 0 & m \neq n \end{cases} \qquad U \sim \mathcal{U}(0, 1).
$$

For several parameters, the basis functions are products of one-dimensional polynomials, one per parameter, for example $\Psi(u) = \psi_2(u_1)\,\psi_1(u_3)$. Each basis function is labelled by its multi-index $\alpha = (\alpha_1, \ldots, \alpha_d)$, the degree in each parameter; this example has $\alpha = (2, 0, 1)$. With the **total degree** $p$, all products with $\alpha_1 + \dots + \alpha_d \le p$ are used.

In [ ]:
t = range(0, 1; length = 200)
multi_indices = reshape(collect(0:4), :, 1)   # one parameter, degrees 0 to 4
Psi_1d = BattMo.pce_design_matrix(reshape(collect(t), :, 1), multi_indices)

fig_legendre = Figure(size = (600, 350))
ax = Axis(fig_legendre[1, 1]; xlabel = "u", ylabel = "ψₙ(u)", title = "Orthonormal Legendre polynomials on [0, 1]")
for n in 0:4
    lines!(ax, t, Psi_1d[:, n + 1]; label = "n = $n", linewidth = 2)
end
axislegend(ax; position = :ct, orientation = :horizontal)
fig_legendre

### Why orthonormality gives the Sobol indices for free

Write the surrogate as $\hat Y = \sum_\alpha c_\alpha \Psi_\alpha(u)$. Because the basis functions are orthonormal, the mean of $\hat Y$ is the constant coefficient $c_0$, and its variance is simply the **sum of the squared coefficients** of all other terms:

$$
\operatorname{Var}(\hat Y) = \sum_{\alpha \neq 0} c_\alpha^2 .
$$

Each term belongs to the group of parameters it depends on, so we only need to sort the squared coefficients into groups. With two parameters and degree 2:

| Term | Depends on | Counts towards |
| --- | --- | --- |
| $c_{(1,0)}\,\psi_1(u_1)$, $c_{(2,0)}\,\psi_2(u_1)$ | $u_1$ only | $S_1$ and $S_{T,1}$ |
| $c_{(0,1)}\,\psi_1(u_2)$, $c_{(0,2)}\,\psi_2(u_2)$ | $u_2$ only | $S_2$ and $S_{T,2}$ |
| $c_{(1,1)}\,\psi_1(u_1)\psi_1(u_2)$ | $u_1$ and $u_2$ (interaction) | $S_{T,1}$ and $S_{T,2}$ only |

So $S_i$ is the sum of $c_\alpha^2$ over terms that depend on $u_i$ *only*, and $S_{T,i}$ the sum over *all* terms that depend on $u_i$, each divided by the total variance. These are the exact indices of the polynomial, so they always satisfy $0 \le S_i \le S_{T,i} \le 1$.

### Fitting the polynomial

The surrogate has $P = \binom{d + p}{p}$ coefficients, for example 56 for $d = 5$ parameters and degree $p = 3$. To fit them, BattMo:

1. Chooses $N$ sample points with a **Latin hypercube** design [McKay et al. 1979]. The range of each parameter is divided into $N$ equal intervals, and every interval receives exactly one point. This spreads the points more evenly than purely random sampling (see the plot below).
2. Runs the model at these points.
3. Finds the coefficients by **least squares**, the polynomial that best matches the model outputs. This requires more points than coefficients; the default is $N = 2P$.

In [ ]:
rng = Random.Xoshiro(7)
n_points = 10
U_random = rand(rng, n_points, 2)
U_lhs = BattMo.latin_hypercube(rng, n_points, 2)

fig_lhs = Figure(size = (700, 350))
for (k, (U_plot, title)) in enumerate(((U_random, "Random"), (U_lhs, "Latin hypercube")))
    ax = Axis(fig_lhs[1, k]; title, aspect = 1, xlabel = "u₁", ylabel = k == 1 ? "u₂" : "",
        xticks = 0:0.2:1, yticks = 0:0.2:1, limits = (0, 1, 0, 1))
    vlines!(ax, (0:n_points) ./ n_points; color = (:black, 0.12))
    hlines!(ax, (0:n_points) ./ n_points; color = (:black, 0.12))
    scatter!(ax, U_plot[:, 1], U_plot[:, 2]; markersize = 12)
end
fig_lhs

In the Latin hypercube, every row and every column of the grid contains exactly one point; the random sample leaves gaps and clusters.

### Is the surrogate good enough? The $Q^2$ check

The PCE indices are only as good as the polynomial. BattMo measures this with the **leave-one-out** coefficient of determination $Q^2$ [Blatman & Sudret 2010]: for each simulation, imagine fitting the polynomial *without* it and predicting it. $Q^2$ is one minus the mean squared prediction error divided by the variance of the outputs. $Q^2 = 1$ means perfect predictions of unseen simulations; $Q^2 = 0$ means the surrogate is no better than predicting the mean. Thanks to a classical identity for least squares, this needs only one fit, not $N$.

BattMo prints a warning when $Q^2 < 0.9$. The Ishigami function contains sines and a fourth power, which a low-degree polynomial cannot represent well, so it is a good test of the check:

In [ ]:
d = 3
println("Degree     P     N      Q²    ST estimates (exact: ", join(round.(exact_ST; digits = 3), ", "), ")")
for degree in (3, 5, 7, 9)
    P = binomial(d + degree, degree)
    N = 2P
    rng = Random.Xoshiro(42)
    U_pce = BattMo.latin_hypercube(rng, N, d)
    Y_pce = [ishigami(row) for row in eachrow(U_pce)]
    est = BattMo.estimate_pce_sobol_indices(U_pce, reshape(Y_pce, :, 1); degree)
    @printf("%6d  %4d  %4d  %6.3f    %s\n", degree, P, N, est.pce.q2[1], join(round.(est.total_order[:, 1]; digits = 3), ", "))
end

With degrees 3 and 5, $Q^2$ is even negative: the polynomial predicts unseen points worse than simply guessing the mean value. The indices are clearly wrong, and the check correctly tells us not to trust them. At degree 7, $Q^2 \approx 0.96$ and the indices are already close; at degree 9, $Q^2 \approx 0.997$ and the indices match the exact values to about 0.01 with 440 evaluations. The Saltelli scheme needed over 5000 evaluations for a worse accuracy.

Quantities of a battery simulation, such as the energy delivered in a discharge, usually depend much more smoothly on the parameters than the Ishigami function, and a degree of 2 to 4 is often enough. **Always check $Q^2$.**

The confidence intervals of the PCE indices are computed with a *residual bootstrap*: the misfit of the polynomial is resampled, the polynomial is refitted many times, and the spread of the indices is reported. These intervals only make sense when $Q^2$ is close to one.

### Which method when?

| | Saltelli (`method = :saltelli`, default) | PCE (`method = :pce`) |
| --- | --- | --- |
| Number of simulations | $N(d + 2)$, typically hundreds to thousands | $2\binom{d+p}{p}$ by default, e.g. 112 for $d = 5$, $p = 3$ |
| Assumptions | None | Output is well approximated by a polynomial |
| Error | Decreases like $1/\sqrt{N}$ | Depends on how well the polynomial fits |
| Quality check | Width of the confidence intervals | $Q^2$, and the confidence intervals |
| Best for | Non-smooth outputs (kinks, jumps), reference results | Smooth outputs and expensive simulations |

## 6. A sensitivity analysis of a battery discharge

We now apply both methods to a 1C constant-current discharge of the cell parameterised by Chen et al. [Chen et al. 2020], an LG M50 cylindrical cell with an NMC811 positive electrode and a graphite–silicon negative electrode.

### The base case

The workflow follows the calibration API: create a `Simulation`, wrap it in a `SobolSensitivity`, declare which parameters vary, and `solve`.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
cycling_protocol["DRate"] = 1.0
model = LithiumIonBattery()
sim = Simulation(model, cell_parameters, cycling_protocol)

sa = SobolSensitivity(sim)

### Choosing the uncertain parameters and their ranges

We study five parameters that are notoriously hard to measure:

- the **solid diffusion coefficients** of both electrodes, which control how fast lithium moves inside the active material particles;
- the **reaction rate constants** of both electrodes, which control how easily lithium crosses the particle surface (the charge transfer kinetics);
- the **Bruggeman coefficient**, which controls how much the tortuous pore structure slows down transport in the electrolyte.

A parameter is identified by its path in the cell parameters. Its nominal value must lie within its bounds. Diffusion coefficients and rate constants are uncertain by a *factor* rather than by an absolute amount, so we let them vary over a factor of 3 or 10 around their nominal values, and sample them uniformly on a logarithmic scale with `scaling = :log`. Otherwise almost all samples would land in the upper decade of the range.

In [ ]:
for electrode in ["NegativeElectrode", "PositiveElectrode"]
    D = cell_parameters[electrode]["ActiveMaterial"]["DiffusionCoefficient"]
    k0 = cell_parameters[electrode]["ActiveMaterial"]["ReactionRateConstant"]
    free_sensitivity_parameter!(
        sa, [electrode, "ActiveMaterial", "DiffusionCoefficient"];
        lower_bound = D / 3, upper_bound = 3 * D, scaling = :log,
    )
    free_sensitivity_parameter!(
        sa, [electrode, "ActiveMaterial", "ReactionRateConstant"];
        lower_bound = k0 / 10, upper_bound = 10 * k0, scaling = :log,
    )
end

free_sensitivity_parameter!(
    sa, ["Electrolyte", "BruggemanCoefficient"];
    lower_bound = 1.3, upper_bound = 1.6,
)

The Bruggeman range is kept moderate on purpose. For larger values the discharge becomes limited by electrolyte depletion, and the moment the cut-off voltage is reached starts to jump erratically between neighbouring parameter sets. Such jumps add variance that no method can attribute to a parameter in a meaningful way.

### Choosing the output quantities

Sobol indices are computed for *scalar* outputs. By default BattMo uses:

- `Energy`: the energy delivered during the discharge, in Wh;
- `Power`: the average power, in W;
- `Heat`: the irreversible heat generated, in Wh, computed from the energy balance of Bernardi et al. [Bernardi et al. 1985].

Any function of the simulation output can be added. Here we add the peak heat generation rate, in W:

In [ ]:
sa.quantities["PeakHeatRate"] = output -> maximum(compute_heat_generation(output).rate)

print_sensitivity_overview(sa)

### How long will this take?

Before starting a study, it is worth estimating its cost. We time one simulation of the base case; the first run includes compilation, so we time the second one. The Saltelli scheme needs $N(d + 2)$ simulations, the PCE $2\binom{d + p}{p}$ by default.

We use $N = 32$ here to keep the run time short; a production study would use a larger $N$.

In [ ]:
N = 32
degree = 3
d = length(sa.parameter_targets)
n_saltelli = N * (d + 2)
n_pce = 2 * binomial(d + degree, degree)

format_time(t) = t < 120 ? "$(round(t; digits = 1)) s" : "$(round(t / 60; digits = 1)) min"

solve(sim; info_level = -1)
time_single = @elapsed solve(sim; info_level = -1)

println("One simulation: $(format_time(time_single))")
println("Saltelli:       $n_saltelli simulations, about $(format_time(n_saltelli * time_single)) serial")
println("PCE:            $n_pce simulations, about $(format_time(n_pce * time_single)) serial")
println("Julia threads:  $(Threads.nthreads())")

### Method 1: Saltelli

The simulations run in parallel when Julia has several threads. We pass a seeded random number generator so that the results are reproducible.

In [ ]:
time_saltelli = @elapsed result = solve(sa; N, rng = Random.Xoshiro(42))
println("Saltelli: $(format_time(time_saltelli))")

print_sobol_indices(result)

`print_sobol_indices` prints one table per quantity, with the parameters sorted by their total-order index and the 95% bootstrap confidence intervals. A plot makes the comparison across quantities easier. In each panel, the blue bars are the total-order indices and the orange bars the first-order indices; the whiskers are the confidence intervals.

In [ ]:
function parameter_label(name)
    parts = split(name, ".")
    owner = Dict("NegativeElectrode" => "NE", "PositiveElectrode" => "PE")
    field = replace(parts[end], r"(?<=[a-z])(?=[A-Z])" => " ")
    return "$(get(owner, parts[1], parts[1])) $field"
end

labels = parameter_label.(result.parameter_names)
q = length(result.quantity_names)
order = sortperm(vec(sum(result.total_order; dims = 2)))   # most influential parameters at the top
color_ST, color_S1 = "#2a78d6", "#eb6834"

fig = Figure(size = (1000, 640))
for (k, quantity) in enumerate(result.quantity_names)
    row, col = fldmod1(k, 2)
    ax = Axis(fig[row, col]; title = quantity, xlabel = "Sobol index", yticks = (1:d, labels[order]), ygridvisible = false)
    col > 1 && hideydecorations!(ax)
    for (offset, index, ci, color) in ((0.19, result.total_order, result.total_order_ci, color_ST), (-0.19, result.first_order, result.first_order_ci, color_S1))
        y = (1:d) .+ offset
        barplot!(ax, y, index[order, k]; direction = :x, width = 0.36, color)
        rangebars!(ax, y, ci[1][order, k], ci[2][order, k]; direction = :x, color = (:black, 0.55), whiskerwidth = 6)
    end
    vlines!(ax, 0; color = :black, linewidth = 1)
end
Legend(
    fig[cld(q, 2) + 1, :], [PolyElement(color = color_ST), PolyElement(color = color_S1)],
    ["Total order ST (with interactions)", "First order S1 (alone)"]; orientation = :horizontal, framevisible = false,
)
Label(fig[0, :], "Sobol indices (Saltelli), N = $(result.N)"; fontsize = 18, font = :bold)
fig

How to read this figure, step by step:

1. **Find the long blue bars.** These parameters drive most of the variation of the quantity. If you want a more reliable prediction of that quantity, these are the parameters to measure more accurately.
2. **Find the parameters with blue bars close to zero.** Within the chosen ranges they hardly influence the quantity, and they can be fixed at their nominal values, for example before a calibration.
3. **Compare the blue and orange bars of each parameter.** If they are about equal, the parameter acts mostly on its own. A blue bar that is clearly longer than the orange one points to interactions with other parameters.
4. **Compare the panels.** The ranking can differ from one quantity to another: a parameter that is irrelevant for the energy can matter for the heat generation.
5. **Look at the whiskers.** With only $N = 32$, the confidence intervals are wide, small indices can be slightly negative, and an orange bar can even be longer than the blue bar, which is impossible for the exact indices. This is sampling noise. The ranking of the most influential parameters is usually already reliable, but small indices need a larger $N$ to be resolved.

### Method 2: PCE

With `method = :pce`, BattMo samples a Latin hypercube, fits a polynomial of the given total degree, and prints $Q^2$ in the title of each table. We use a copy of the analysis object so that the Saltelli result stored in `sa.result` is kept.

In [ ]:
sa_pce = deepcopy(sa)
time_pce = @elapsed result_pce = solve(sa_pce; method = :pce, degree, rng = Random.Xoshiro(42))
println("PCE: $(format_time(time_pce))")

print_sobol_indices(result_pce)

### Comparing the two methods

Each panel shows one index of one quantity for both methods, with the confidence intervals as whiskers. Where the intervals overlap, the methods agree within their uncertainty.

In [ ]:
println(rpad("Method", 10), rpad("Simulations", 13), rpad("Time", 10), "Q² (", join(result_pce.quantity_names, ", "), ")")
println(rpad("Saltelli", 10), rpad(size(result.samples, 1), 13), rpad(format_time(time_saltelli), 10), "-")
println(rpad("PCE", 10), rpad(size(result_pce.samples, 1), 13), rpad(format_time(time_pce), 10), join(round.(result_pce.pce.q2; digits = 4), ", "))

methods = (
    (0.14, result, color_ST, :circle, "Saltelli ($(size(result.samples, 1)) simulations)"),
    (-0.14, result_pce, color_S1, :diamond, "PCE degree $degree ($(size(result_pce.samples, 1)) simulations)"),
)
indices = (("Total order ST", r -> r.total_order, r -> r.total_order_ci), ("First order S1", r -> r.first_order, r -> r.first_order_ci))

fig_compare = Figure(size = (1100, 620))
compare_axes = Matrix{Axis}(undef, length(indices), q)
for (row, (index_label, index, ci)) in enumerate(indices), (k, quantity) in enumerate(result.quantity_names)
    ax = compare_axes[row, k] = Axis(
        fig_compare[row, k]; title = row == 1 ? quantity : "", ylabel = k == 1 ? index_label : "",
        xlabel = row == length(indices) ? "Sobol index" : "", yticks = (1:d, labels[order]), ygridvisible = false,
    )
    k > 1 && hideydecorations!(ax; label = false)
    vlines!(ax, 0; color = :black, linewidth = 1)
    for (offset, r, color, marker, _) in methods
        y = (1:d) .+ offset
        lower, upper = ci(r)
        rangebars!(ax, y, lower[order, k], upper[order, k]; direction = :x, color, linewidth = 2, whiskerwidth = 6)
        scatter!(ax, index(r)[order, k], y; color, marker, markersize = 11, strokecolor = :white, strokewidth = 1.5)
    end
end
linkxaxes!(compare_axes...)
Legend(
    fig_compare[length(indices) + 1, :],
    [MarkerElement(; color, marker, markersize = 11) for (_, _, color, marker, _) in methods],
    [label for (_, _, _, _, label) in methods]; orientation = :horizontal, framevisible = false,
)
fig_compare

For the energy, the average power and the heat, $Q^2$ is typically close to one for this setup: these are *integrals* over the discharge, which depend smoothly on the parameters. The PCE estimates then agree with the Saltelli estimates, with narrower intervals from half the simulations.

The peak heat generation rate behaves differently. It is a *maximum over time*: when a parameter change moves the moment of the peak from one part of the discharge to another, the peak value changes abruptly. A polynomial cannot follow such kinks, so $Q^2$ is low, BattMo prints a warning, and the PCE indices of this quantity should not be trusted. For such a quantity, use the Saltelli scheme, with a larger $N$.

This is exactly the trade-off of section 5: the PCE is the better choice when simulations are expensive and the quantity is smooth, and the Saltelli scheme is the safer choice when it is not. Check $Q^2$ before using PCE results.

### Scatter plots: *how* the parameters act

The indices tell us *how much* each parameter matters; scatter plots of the simulations show *how*. Just like for the Ishigami function in section 2, every Saltelli simulation is a point, and every column spans the range of one parameter (on a log axis for the log-scaled ones). A clear trend in a column means a large first-order effect; the vertical spread around the trend comes from the other parameters.

In [ ]:
targets = [sa.parameter_targets[String.(split(name, "."))] for name in result.parameter_names]
units = Dict("Energy" => "Wh", "Power" => "W", "Heat" => "Wh", "PeakHeatRate" => "W")

fig_scatter = Figure(size = (220 * d, 190 * q))
columns = reverse(order)
axes = [
    Axis(
        fig_scatter[k, j]; xscale = targets[i].scaling == :log ? log10 : identity,
        limits = ((targets[i].vmin, targets[i].vmax), nothing), xlabel = labels[i],
        ylabel = "$quantity / $(units[quantity])", xticklabelsvisible = false,
    )
        for (k, quantity) in enumerate(result.quantity_names), (j, i) in enumerate(columns)
]
for (k, quantity) in enumerate(result.quantity_names), (j, i) in enumerate(columns)
    scatter!(axes[k, j], result.samples[:, i], result.values[:, k]; color = (color_ST, 0.45), markersize = 6)
    k < q && hidexdecorations!(axes[k, j]; grid = false)
    j > 1 && hideydecorations!(axes[k, j]; grid = false)
end
foreach(row -> linkyaxes!(row...), eachrow(axes))
fig_scatter

Read the scatter plots as in section 2:

- A column with a clear upward or downward trend belongs to a parameter with a large first-order index. The shape of the trend shows whether the effect is roughly linear or saturates, for example when a diffusion coefficient stops being limiting above a certain value.
- A column without a trend belongs to a parameter that hardly matters on its own. If the *spread* of the points changes along the column, the parameter interacts with others, like $X_3$ of the Ishigami function.
- Isolated outliers, or clouds that split into separate bands, point to a non-smooth response, for example a discharge that stops at a different cut-off condition. Such points inflate the variance, and both methods attribute it to the parameters. If you see them, consider narrowing the ranges.

The scatter plots reuse the stored simulations (`result.samples` and `result.values`), so they cost nothing extra. It is good practice to look at them for every study.

## 7. Practical tips and pitfalls

- **Choose the ranges with care.** The indices answer the question *"given these ranges"*. A parameter with a range that is too wide looks more important than it is, and vice versa. Base the ranges on measurement uncertainty, spread in the literature, or the range you want to explore.
- **Use `scaling = :log`** for parameters that are uncertain by a factor, such as diffusion coefficients, rate constants and conductivities.
- **Start small.** Time one simulation, run a small study (for example a PCE of degree 2, or Saltelli with $N = 16$), look at the results and the scatter plots, and only then invest in a large run.
- **Check the confidence intervals** (Saltelli) and **$Q^2$** (PCE). Wide intervals mean more samples; a low $Q^2$ means a higher degree, more samples, or the Saltelli scheme.
- **Watch out for non-smooth outputs.** Maxima over time, or quantities that depend on *when* a cut-off is reached, can jump between neighbouring parameter sets. Scatter plots reveal this.
- **Failed simulations** give `NaN` and are dropped; `result.n_valid` reports how many samples were used. If many fail, narrow the ranges or adjust the solver settings, as the remaining samples no longer cover the parameter space evenly.
- **Parameters are assumed independent and uniformly distributed.** Correlated parameters (for example a diffusion coefficient fitted together with a particle radius) are not supported.
- **Reproducibility:** pass a seeded random number generator, `rng = Random.Xoshiro(42)`.
- **Fix the unimportant parameters.** A typical use is to screen many parameters, fix those with a small total-order index, and then calibrate only the remaining ones.

## 8. Summary

- Global sensitivity analysis varies all uncertain parameters at once over their full ranges, and attributes the variance of an output to the parameters.
- The **first-order index** $S_i$ is the share of the variance explained by $X_i$ alone; the **total-order index** $S_{T,i}$ adds all interactions involving $X_i$. A small $S_{T,i}$ means the parameter can be fixed.
- The **Saltelli scheme** estimates both indices from $N(d + 2)$ simulations, with no assumption on the model, but converges slowly ($\propto 1/\sqrt{N}$).
- A **polynomial chaos expansion** fits a polynomial surrogate and reads the indices off its coefficients. It needs far fewer simulations for smooth outputs; check $Q^2$.
- In BattMo: `SobolSensitivity(sim)`, `free_sensitivity_parameter!`, `sa.quantities`, `solve(sa; method = :saltelli or :pce)`, `print_sobol_indices`.

## References and further reading

**Papers on which BattMo's implementation is based**

- I. M. Sobol' (2001). *Global sensitivity indices for nonlinear mathematical models and their Monte Carlo estimates.* Mathematics and Computers in Simulation 55, 271–280. [doi:10.1016/S0378-4754(00)00270-6](https://doi.org/10.1016/S0378-4754(00)00270-6). The definition of the indices and the variance decomposition.
- A. Saltelli, P. Annoni, I. Azzini, F. Campolongo, M. Ratto, S. Tarantola (2010). *Variance based sensitivity analysis of model output. Design and estimator for the total sensitivity index.* Computer Physics Communications 181, 259–270. [doi:10.1016/j.cpc.2009.09.018](https://doi.org/10.1016/j.cpc.2009.09.018). The sampling scheme with $A$, $B$ and $A_B^{(i)}$, and the first-order estimator.
- M. J. W. Jansen (1999). *Analysis of variance designs for model output.* Computer Physics Communications 117, 35–43. [doi:10.1016/S0010-4655(98)00154-4](https://doi.org/10.1016/S0010-4655(98)00154-4). The total-order estimator.
- B. Sudret (2008). *Global sensitivity analysis using polynomial chaos expansions.* Reliability Engineering & System Safety 93, 964–979. [doi:10.1016/j.ress.2007.04.002](https://doi.org/10.1016/j.ress.2007.04.002). Sobol indices from the coefficients of a PCE.
- G. Blatman, B. Sudret (2010). *An adaptive algorithm to build up sparse polynomial chaos expansions for stochastic finite element analysis.* Probabilistic Engineering Mechanics 25, 183–197. [doi:10.1016/j.probengmech.2009.10.003](https://doi.org/10.1016/j.probengmech.2009.10.003). The leave-one-out error $Q^2$.
- M. D. McKay, R. J. Beckman, W. J. Conover (1979). *A comparison of three methods for selecting values of input variables in the analysis of output from a computer code.* Technometrics 21, 239–245. [doi:10.1080/00401706.1979.10489755](https://doi.org/10.1080/00401706.1979.10489755). Latin hypercube sampling.
- D. Bernardi, E. Pawlikowski, J. Newman (1985). *A general energy balance for battery systems.* Journal of The Electrochemical Society 132, 5–12. [doi:10.1149/1.2113792](https://doi.org/10.1149/1.2113792). The heat generation used for the `Heat` quantity.

**Test function and battery data**

- T. Ishigami, T. Homma (1990). *An importance quantification technique in uncertainty analysis for computer models.* Proceedings of ISUMA '90, 398–403. [doi:10.1109/ISUMA.1990.151285](https://doi.org/10.1109/ISUMA.1990.151285).
- C.-H. Chen, F. Brosa Planella, K. O'Regan, D. Gastol, W. D. Widanage, E. Kendrick (2020). *Development of experimental techniques for parameterization of multi-scale lithium-ion battery models.* Journal of The Electrochemical Society 167, 080534. [doi:10.1149/1945-7111/ab9050](https://doi.org/10.1149/1945-7111/ab9050).

**A gentle textbook**

- A. Saltelli, M. Ratto, T. Andres, F. Campolongo, J. Cariboni, D. Gatelli, M. Saisana, S. Tarantola (2008). *Global Sensitivity Analysis: The Primer.* Wiley. [doi:10.1002/9780470725184](https://doi.org/10.1002/9780470725184). An accessible introduction to the whole field, written for practitioners.

**In the BattMo documentation**

- The user guide page *Sobol sensitivity analysis* (`docs/src/manuals/user_guide/sobol_sensitivity.md`) documents all options, the estimators and their implementation details.
- The script `examples/example_sobol.jl` contains the battery study of section 6 as a standalone example.